In [ ]:
import sys
from pathlib import Path

scripts_dir = Path.cwd().parent.parent / "scripts"
sys.path.insert(0, str(scripts_dir))

In [ ]:
# Path to test .mrxs
path_CMU1 = r"E:\Christine\testdata\CMU-1.mrxs"
path_CMU3 = r"E:\Christine\testdata\CMU-3.mrxs"
zarr_dir = r"E:\Christine\testdata\zarr"
cache_tissue = r"E:\Christine\testdata\cache_tissue_artifact.pkl"
cache_features = r"E:\Christine\testdata\feature_summary.csv"

slides = [path_CMU1, path_CMU3]

In [ ]:
from tissue_artifact_segmentation import SegmentMany

segmenter = SegmentMany(slides, cache_tissue, zarr_dir, "tissue", version="default")

In [ ]:
# Visualize Single Slide
import os
from wsidata import open_wsi

zarr_path = os.path.join(zarr_dir, os.path.basename(path_CMU3).replace(".mrxs", ".zarr"))
wsi = open_wsi(path_CMU3, zarr_path)
wsi

In [ ]:
from feature_extraction import ExtractMany

model = "h-optimus-0"
extractor = ExtractMany(slides, cache_features, zarr_dir, model, remove_artifacts = False)

In [ ]:
# Generate df that resembles pathology df in 100k project
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "filename": [str(x) for x in slides],
    "class":  np.random.randint(0, 2, size=len(slides))
}).reset_index(drop=True)

times = 10
df_expanded = pd.DataFrame(np.repeat(df.values, times, axis=0), columns=df.columns)

print(df_expanded)

In [ ]:
from abmil import TrainABMILPipeline

abmil_path = r"E:\Christine\testdata\abmil.pt"

pipeline = TrainABMILPipeline(df_expanded, 'filename', 'class', 'features_h-optimus-0', 'tiles_224', zarr_dir, abmil_path)
pipeline.run_pipeline(max_tiles=50000, n_epochs=10, seed=42, validation_fraction=0.10, early_stopping_patience=5)

In [ ]:
from abmil import ABMILInference

abmil_path = r"E:\Christine\testdata\abmil.pt"
inference_path = r"E:\Christine\testdata\inference.pkl"

inference = ABMILInference(checkpoint_path=abmil_path, zarr_dir=zarr_dir, slides=slides, cache_path=inference_path)
inference.process_slides(validate = True)

In [ ]:
from roi_selection import ROISelector

abmil_path = r"E:\Christine\testdata\abmil.pt"
inference_path = r"E:\Christine\testdata\inference.pkl"

selector = ROISelector(cache_path = inference_path, slide_path = path_CMU1, top_k = 20, bottom_k = 10)
selector.tiles_to_cut()

In [ ]:
sdata_lmd = selector.get_sdata_lmd()
top_polygons, bottom_polygons = selector.napari_polygons()

In [ ]:
import napari
from napari_wsi.backends.openslide import OpenSlideStore

viewer = napari.Viewer()
store = OpenSlideStore(path_CMU1, color_space="sRGB")
(layer,) = store.to_viewer(viewer, spatial_transform=False)

viewer.add_shapes(
    top_polygons,
    shape_type="polygon",
    edge_color="red",
    face_color="transparent",
    edge_width=30,
    name="top_polygons",
)
viewer.add_shapes(
    bottom_polygons,
    shape_type="polygon",
    edge_color="blue",
    face_color="transparent",
    edge_width=30,
    name="bottom_polygons",
)

napari.run()

In [ ]:
sdata_lmd

In [ ]:
# Manually in Napari: 

# 1) Add calibration points (3 points, easy to locate on LMD)
# File -> new layer -> points 

# 2) Add square (to check correct placement of shapes on LMD)
# File -> new layer -> shapes

In [ ]:
# Add manually selected calibration points to sdata.points["calibration_points"]

from spatialdata.models import PointsModel
import numpy as np

points_layer = viewer.layers["calibration_points"]
image_points = points_layer.data
print(image_points)

sdata_lmd.points["calibration_points"] = PointsModel.parse(
    np.array(image_points)
)

In [ ]:
# Add square to ensure correct placement of polygons

from spatialdata.models import ShapesModel
from shapely.geometry import Polygon
import geopandas as gpd

square_layer = viewer.layers["square"]
image_square = square_layer.data
print(image_square)

polygons = [Polygon(coords) for coords in image_square]
square_gdf = gpd.GeoDataFrame(
    {"shape_id": [f"square_{i}" for i in range(len(polygons))]},
    geometry=polygons
)

sdata_lmd.shapes["square"] = ShapesModel.parse(square_gdf)

In [ ]:
sdata_lmd

In [ ]:
H = sdata_lmd.images["wsi_thumbnail"].data.shape[1]
print(H)

In [ ]:
import geopandas as gpd
from spatialdata.models import ShapesModel

def add_square_to_annotation(annotation_gdf, square_gdf):
    combined = gpd.GeoDataFrame(
        pd.concat([annotation_gdf, square_gdf], ignore_index=True),
        geometry="geometry",
        crs=annotation_gdf.crs,
    )
    return ShapesModel.parse(combined)


In [ ]:
annotation_top = add_square_to_annotation(
    sdata_lmd.shapes["top_tiles"],
    sdata_lmd.shapes["square"],
)

annotation_bottom = add_square_to_annotation(
    sdata_lmd.shapes["bottom_tiles"],
    sdata_lmd.shapes["square"],
)


In [ ]:
from dvpio.write import write_lmd
import os

lmd_dir = r"E:\Christine\testdata\lmd"

path_lmd = os.path.join(lmd_dir, "CMU1.xml")

# Transform coordinates from napari to LMD coordinate system
affine_transformation = np.array([
    [1,  0, 0],
    [0, -1, H],
    [0,  0, 1]
])

# Write LMD file with tiles and calibration points
write_lmd(
    path = path_lmd,
    annotation = annotation_top,
    calibration_points=sdata_lmd.points["calibration_points"],
    affine_transformation=affine_transformation
)